In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
from datetime import datetime, date
import re

In [2]:
url = "https://www.stats.gov.cn/sj/fbrc/bnxxfb/"

response = requests.get(url)
response.encoding = 'utf-8'

In [3]:
if response.status_code == 200:
    # 解析 HTML 內容
    soup = BeautifulSoup(response.text, 'html.parser')
    
    # 查找目標數據（假設 "Press Conference on Economic Situation" 位於表格的第一行第一列）
    table = soup.find('table')  # 假設頁面只有一個表格
    if table:
        # 查找第一個行
        first_row = table.find('tr')
        if first_row:
            # 查找第一個列
            first_cell = first_row.find('td')
            if first_cell:
                target_text = first_cell.get_text(strip=True)
                print("Press Conference on Economic Situation:", target_text)
            else:
                print("第一列的第一個單元格未找到")
        else:
            print("表格的第一行未找到")
    else:
        print("表格未找到")
else:
    print(f"請求失敗，狀態碼: {response.status_code}")

Press Conference on Economic Situation: 序号


### 國家統計局

In [4]:
# 目標 URL
url = "https://www.stats.gov.cn/sj/fbrc/bnxxfb/"

# 發送 HTTP GET 請求
response = requests.get(url)
# 設置正確的編碼，國家統計局頁面常見 GB18030 或 utf-8，依實際回傳為主
response.encoding = 'utf-8'  

# 檢查請求是否成功
if response.status_code == 200:
    # 解析 HTML 內容
    soup = BeautifulSoup(response.text, 'html.parser')
    
    # 查找表格
    table = soup.find('table')
    if table:
        # 提取表格的所有行
        rows = table.find_all('tr')
        
        # 提取第一行作為欄位名稱 (假設第一行就是標題行)
        header_cells = rows[0].find_all('td')
        headers = [cell.get_text(strip=True) for cell in header_cells]
        
        # 初始化一個空的數據列表
        data = []
        
        # 提取其餘行的數據
        for row in rows[1:]:
            cells = row.find_all('td')
            row_data = [cell.get_text(strip=True) for cell in cells]
            # 確保行不為空
            if any(cell_val.strip() for cell_val in row_data):
                data.append(row_data)
        
        # 若最後一行真的多餘 (常見狀況)，可視情況保留或刪除
        data = data[:-1]
        
        # 將數據轉換為 DataFrame，並設置欄位名稱
        df = pd.DataFrame(data, columns=headers)

        # 依照先前邏輯：偶數行(0-based) → 內容行；奇數行 → 時間行
        # 如果你確定表格結構：0 -> 內容行, 1 -> 時間行, 2-> 內容行, 3-> 時間行...
        content_date_row = df.iloc[0::2, :].reset_index(drop=True)
        time_row = df.iloc[1::2, 0].reset_index(drop=True)
        
        # 處理序號欄位（移除前綴的數字及點）
        content_date_row['序号'] = content_date_row['序号'].apply(
            lambda x: re.sub(r'^\d+\.\s*', '', x)
        )
        
        # 刪除欄位名稱尾巴可能出現的句點 (如 "3月." → "3月")
        content_date_row.columns = [
            re.sub(r'\.$', '', col) for col in content_date_row.columns
        ]
        
        # 只處理這些「月份欄位」
        valid_months = ['1月', '2月', '3月', '4月', '5月', '6月', '7月', '8月', '9月', '10月', '11月', '12月']
        
        # 月份對應
        month_mapping = {
            '1月': '01', '2月': '02', '3月': '03', '4月': '04',
            '5月': '05', '6月': '06', '7月': '07', '8月': '08',
            '9月': '09', '10月': '10', '11月': '11', '12月': '12'
        }
        
        # 對於每一筆資料，針對各月份欄位進行處理
        for i in range(len(content_date_row)):
            for col in content_date_row.columns:
                # 跳過非「月份」的欄位 (例如 '序号', '内容')
                if col not in valid_months:
                    continue
                
                cell_val = content_date_row.at[i, col]
                
                # 如果是 "……" 或是空，就用 NaN
                if not cell_val or cell_val.strip() == "……":
                    content_date_row.at[i, col] = np.nan
                    continue
                
                # 組合字串 => "2025/3月/17/五 - 9:30" 這類
                # time_row[i] 是對應的時間(例如 "9:30", "10:00")
                # 若超出範圍則給預設時間 "00:00"
                time_str = time_row[i] if i < len(time_row) else "00:00"
                date_time_str = f"2025/{col}/{cell_val} - {time_str}"
                
                # Debug (可選): 看看每次組合出什麼
                # print("Before regex:", date_time_str)
                
                # 將月份「x月」轉成「xx」
                date_time_str = date_time_str.replace(col, month_mapping[col])
                
                # 只保留日期前面一到兩位數字 -> "17/五" 變 "17"
                # 這裡的寫法：針對「數字 + / + (任何中文字)」進行替換
                # 若你還有 "17/五註53" 這種複雜情形，也會被替換到「17」
                date_time_str = re.sub(r'(\d{1,2})/[\u4e00-\u9fa5]+', r'\1', date_time_str)
                # 說明：
                #  - (\d{1,2})：抓一到兩位數字（日期）
                #  - /[\u4e00-\u9fa5]+.*：抓 "/" 以及至少一個中文字 + 其後所有字
                #  - r'\1'：只保留 group(1)，即日期
                
                # 也可能想把 "/其他符號" 全部吃掉，就寫成  re.sub(r'(\d{1,2})/\S+', r'\1', date_time_str)
                # 但要小心別吃掉 "/2025" 之類不該清的
                
                # 把 " - " 改成空格 => "2025/03/17 9:30"
                date_time_str = date_time_str.replace(" - ", " ")
                
                # Debug (可選):
                # print("After regex:", date_time_str)
                
                # 放回 DataFrame
                content_date_row.at[i, col] = date_time_str

        
        # 將「序号」改名為「Indicator」
        content_date_row = content_date_row.rename(columns={'内容': 'Indicator'})
        content_date_row =  content_date_row.drop(columns = '序号')
        
        # 將欄位嘗試轉成日期時間 (若格式不符就變 NaT)
        for col in content_date_row.columns:
            if col in valid_months:  # 只轉換「1月～12月」
                content_date_row[col] = pd.to_datetime(
                    content_date_row[col],
                    format='%Y/%m/%d %H:%M',
                    errors='coerce'
                )
       
        

        # 最後得到想要的 DataFrame
        China_macro_calendar = content_date_row
        

        
    else:
        print("表格未找到")
else:
    print(f"請求失敗，狀態碼: {response.status_code}")

China_macro_calendar

# 如需輸出成 CSV：
# China_macro_calendar.to_csv('China_macro_calendar.csv', index=False)


,Indicator,1月,2月,3月,4月,5月,6月,7月,8月,9月,10月,11月,12月
0,国民经济运行情况,2025-01-17 10:00:00,NaT,2025-03-17 10:00:00,2025-04-16 10:00:00,2025-05-19 10:00:00,2025-06-16 10:00:00,2025-07-15 10:00:00,2025-08-15 10:00:00,2025-09-15 10:00:00,2025-10-20 10:00:00,2025-11-14 10:00:00,2025-12-15 10:00:00
1,2024年国民经济和社会发展统计公报,NaT,2025-02-28 09:30:00,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT
2,季度主要行业增加值初步核算报告,2025-01-18 09:30:00,NaT,NaT,2025-04-17 09:30:00,NaT,NaT,2025-07-16 09:30:00,NaT,NaT,2025-10-21 09:30:00,NaT,NaT
3,采购经理指数月度报告,2025-01-27 09:30:00,NaT,NaT,2025-04-30 09:30:00,2025-05-31 09:30:00,2025-06-30 09:30:00,2025-07-31 09:30:00,2025-08-31 09:30:00,2025-09-30 09:30:00,2025-10-31 09:30:00,2025-11-30 09:30:00,2025-12-31 09:30:00
4,居民消费价格指数月度报告,2025-01-09 09:30:00,2025-02-09 09:30:00,2025-03-09 09:30:00,2025-04-10 09:30:00,2025-05-10 09:30:00,2025-06-09 09:30:00,2025-07-09 09:30:00,2025-08-09 09:30:00,2025-09-10 09:30:00,2025-10-15 09:30:00,2025-11-09 09:30:00,2025-12-10 09:30:00
5,工业生产者价格指数月度报告,2025-01-09 09:30:00,2025-02-09 09:30:00,2025-03-09 09:30:00,2025-04-10 09:30:00,2025-05-10 09:30:00,2025-06-09 09:30:00,2025-07-09 09:30:00,2025-08-09 09:30:00,2025-09-10 09:30:00,2025-10-15 09:30:00,2025-11-09 09:30:00,2025-12-10 09:30:00
6,规模以上工业生产月度报告,2025-01-17 10:00:00,NaT,2025-03-17 10:00:00,2025-04-16 10:00:00,2025-05-19 10:00:00,2025-06-16 10:00:00,2025-07-15 10:00:00,2025-08-15 10:00:00,2025-09-15 10:00:00,2025-10-20 10:00:00,2025-11-14 10:00:00,2025-12-15 10:00:00
7,能源生产情况月度报告,2025-01-17 10:00:00,NaT,2025-03-17 10:00:00,2025-04-16 10:00:00,2025-05-19 10:00:00,2025-06-16 10:00:00,2025-07-15 10:00:00,2025-08-15 10:00:00,2025-09-15 10:00:00,2025-10-20 10:00:00,2025-11-14 10:00:00,2025-12-15 10:00:00
8,固定资产投资（不含农户）月度报告,2025-01-17 10:00:00,NaT,2025-03-17 10:00:00,2025-04-16 10:00:00,2025-05-19 10:00:00,2025-06-16 10:00:00,2025-07-15 10:00:00,2025-08-15 10:00:00,2025-09-15 10:00:00,2025-10-20 10:00:00,2025-11-14 10:00:00,2025-12-15 10:00:00
9,房地产开发和销售情况月度报告,2025-01-17 10:00:00,NaT,2025-03-17 10:00:00,2025-04-16 10:00:00,2025-05-19 10:00:00,2025-06-16 10:00:00,2025-07-15 10:00:00,2025-08-15 10:00:00,2025-09-15 10:00:00,2025-10-20 10:00:00,2025-11-14 10:00:00,2025-12-15 10:00:00


In [5]:
#  寬轉長
China_macro_calendar_long = pd.melt(content_date_row, id_vars=['Indicator'], var_name='Month', value_name='Release_Date')
        
# 移除空值
China_macro_calendar_long = China_macro_calendar_long.dropna(subset=['Release_Date']).reset_index(drop=True)

China_macro_calendar_long = China_macro_calendar_long[['Indicator','Release_Date']]
China_macro_calendar_long.sort_values(by = ['Indicator'], inplace=True)

    
China_macro_calendar_long.to_csv('China_macro_calendar.csv',index=False)
China_macro_calendar_long

,Indicator,Release_Date
15,2024年国民经济和社会发展统计公报,2025-02-28 09:30:00
113,全国居民收支情况季度报告,2025-10-20 10:00:00
76,全国居民收支情况季度报告,2025-07-15 10:00:00
39,全国居民收支情况季度报告,2025-04-16 10:00:00
10,全国居民收支情况季度报告,2025-01-17 10:00:00
...,...,...
31,采购经理指数月度报告,2025-04-30 09:30:00
45,采购经理指数月度报告,2025-05-31 09:30:00
2,采购经理指数月度报告,2025-01-27 09:30:00
82,采购经理指数月度报告,2025-08-31 09:30:00


### United State ALFred

In [ ]:
def fetch_fred_release_times(id_list, year_list):
    base_url = "https://alfred.stlouisfed.org/releases/calendar?pageID=1&ob=rd&od=asc"
    data = []
    
    for rid in id_list:
        for year in year_list:
            url = f"{base_url}&rid={rid}&y={year}&m=0&filtergo=Find+Releases"
            response = requests.get(url)
            if response.status_code != 200:
                print(f"Failed to fetch data for rid={rid}, year={year}")
                continue
            
            soup = BeautifulSoup(response.text, "html.parser")
            table = soup.find("table", class_="table table-condensed table-standard-theme")
            if not table:
                print(f"No table found for rid={rid}, year={year}")
                continue
            
            rows = table.find_all("tr")
            current_date_str = None
            
            for row in rows:
                cols = row.find_all("td")
                
                # 只有 1 欄 -> 通常是日期
                if len(cols) == 1:
                    raw_date = cols[0].get_text(strip=True)
                    # 假如有 "Updated" 字樣，需要去除
                    raw_date = raw_date.replace("Updated", "").strip()
                    current_date_str = raw_date

                # 有 2 欄 -> 通常是 (時間, 指標)
                elif len(cols) == 2:
                    if current_date_str is None:
                        continue  # 若沒有先讀到日期，就跳過

                    time = cols[0].get_text(strip=True)
                    indicator = cols[1].get_text(strip=True)

                    # --- 解析日期 ---
                    # 例如 current_date_str = "Wednesday December 03, 2025"
                    parts = current_date_str.split(" ", 1)
                    if len(parts) < 2:
                        continue
                    date_part = parts[1].strip()  # "December 03, 2025"

                    dt = datetime.strptime(date_part, "%B %d, %Y")  
                    date = dt.strftime("%Y-%m-%d")  # ex: "2025-12-03"

                    # # --- 清理時間 ---
                    # # 若要去除 am/pm，只留 "時:分"
                    # raw_time = raw_time.lower().replace("am", "").replace("pm", "").strip()

                    data.append([indicator,date, time])
                    
                    current_date_str = None  # 用完後清除，以免誤用

    df = pd.DataFrame(data, columns=['indicator','date','time'])
    return df

In [7]:
id_list = [194,435,10,9,101,192,50,54,46]  # 選你要的指標 ID，ID請看指標對應到網址的rid
year_list = [2025]  # 你要查詢的年份

df = fetch_fred_release_times(id_list, year_list)
# df.sort_values(["in"])

df.to_csv("alfred_us_indicator_release_calendar.csv")
df

,indicator,date,time
0,ADP National Employment Report,2025-01-08,7:15 am
1,ADP National Employment Report,2025-02-05,7:15 am
2,ADP National Employment Report,2025-03-05,7:15 am
3,ADP National Employment Report,2025-04-02,7:15 am
4,ADP National Employment Report,2025-04-30,7:15 am
...,...,...,...
92,Producer Price Index,2025-08-14,7:30 am
93,Producer Price Index,2025-09-10,7:30 am
94,Producer Price Index,2025-10-16,7:30 am
95,Producer Price Index,2025-11-14,7:30 am


## Investing

In [30]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException

from bs4 import BeautifulSoup
import pandas as pd
import time


class InvestingCalendarScraper:
    """
    用於爬取 Investing.com 經濟日曆的工具類別。

    主要功能：
      1. 啟動無頭瀏覽器 (Headless Chrome)
      2. 設定時區為 GMT+8
      3. 透過內建 JS 函式篩選特定國家
      4. 透過內建 JS 函式篩選指定日期區間
      5. 無限滾動並點擊「載入更多」以取得完整資料
      6. 將結果解析為 pandas DataFrame
    """

    # ---------------------------- 常數設定 ---------------------------- #
    BASE_URL = "https://www.investing.com/economic-calendar/"
    TIMEZONE_ID_GMT8 = "28"       # 對應 (GMT +8:00) 北京等地
    LOADING_ID = "economicCalendarLoading"
    EVENT_ROW_SELECTOR = "tr.js-event-item"
    SHOW_MORE_SELECTOR = "a.showMoreButton"

    def __init__(
        self,
        start_date: str = None,
        end_date: str = None,
        country_ids: list[str] = None
    ):
        """
        初始化爬蟲參數。

        :param start_date: 開始日期，格式 'YYYY-MM-DD'
        :param end_date:   結束日期，格式 'YYYY-MM-DD'
        :param country_ids: 國家 checkbox 的 value 清單，例如 ['5'] 代表美國
        """
        self.start_date = start_date
        self.end_date = end_date
        self.country_ids = country_ids or []
        # 在初始化時啟動瀏覽器
        self.driver = self._init_driver()

    def _init_driver(self) -> webdriver.Chrome:
        """
        啟動 Headless Chrome 並開啟經濟日曆頁面。
        """
        opts = webdriver.ChromeOptions()
        opts.add_argument("--headless")       # 無頭模式，不顯示 UI
        opts.add_argument("--no-sandbox")     # 避免沙箱問題
        opts.add_argument("--disable-gpu")    # 停用 GPU 加速

        driver = webdriver.Chrome(
            service=Service(ChromeDriverManager().install()),
            options=opts
        )
        driver.get(self.BASE_URL)
        return driver

    def _close_cookie_banner(self) -> None:
        """
        關閉可能出現的 GDPR Cookie 同意橫幅。
        """
        try:
            WebDriverWait(self.driver, 5).until(
                EC.element_to_be_clickable((By.ID, "onetrust-accept-btn-handler"))
            ).click()
        except Exception:
            # 若未出現或無法關閉則忽略
            pass

    def _set_timezone_gmt8(self) -> None:
        """
        將頁面時區設定為 GMT+8。
        透過點擊當前時間元件，開啟下拉列表，再選擇對應 liTz28。
        """
        try:
            # 點擊顯示時區選單
            WebDriverWait(self.driver, 10).until(
                EC.element_to_be_clickable((By.ID, "economicCurrentTime"))
            ).click()
            # 等待選單出現
            WebDriverWait(self.driver, 10).until(
                EC.visibility_of_element_located((By.ID, "economicCurrentTimePop"))
            )
            # 點選對應時區
            self.driver.find_element(By.ID, f"liTz{self.TIMEZONE_ID_GMT8}").click()
            # 等待選單消失
            WebDriverWait(self.driver, 10).until(
                EC.invisibility_of_element_located((By.ID, "economicCurrentTimePop"))
            )
            # 驗證時區顯示已更新
            WebDriverWait(self.driver, 10).until(
                EC.text_to_be_present_in_element(
                    (By.ID, "timeZoneGmtOffsetFormatted"),
                    "(GMT +8:00)"
                )
            )
        except Exception:
            # 若失敗則跳過，不影響後續
            pass

    def _apply_country_filter(self) -> None:
        """
        清除所有國家勾選，然後僅勾選 self.country_ids 中的項目。
        最後呼叫 calendarFilters.innerFiltersSubmit() 套用篩選。
        """
        if not self.country_ids:
            return
        try:
            # 等待 JS 篩選函式載入
            WebDriverWait(self.driver, 10).until(
                lambda d: d.execute_script(
                    "return typeof calendarFilters !== 'undefined' && typeof clearAll === 'function';"
                )
            )
            # 清除所有國家
            self.driver.execute_script("clearAll('country[]');")
            # 勾選指定國家
            for cid in self.country_ids:
                self.driver.execute_script(
                    f"document.getElementById('country{cid}').checked = true;"
                )
            # 提交篩選
            self.driver.execute_script("calendarFilters.innerFiltersSubmit();")
            # 等待資料載入完成
            WebDriverWait(self.driver, 20).until(
                EC.invisibility_of_element_located((By.ID, self.LOADING_ID))
            )
            WebDriverWait(self.driver, 20).until(
                EC.presence_of_all_elements_located((By.CSS_SELECTOR, self.EVENT_ROW_SELECTOR))
            )
        except Exception:
            # 若篩選失敗則忽略
            pass

    def _apply_date_filter(self) -> None:
        """
        若提供 start_date 和 end_date，就呼叫 datePickerFilter 篩選日期區間。
        否則確保初始資料載入。
        """
        if self.start_date and self.end_date:
            try:
                self.driver.execute_script(
                    "calendarFilters.datePickerFilter(arguments[0], arguments[1]);",
                    self.start_date, self.end_date
                )
                WebDriverWait(self.driver, 20).until(
                    EC.invisibility_of_element_located((By.ID, self.LOADING_ID))
                )
            except TimeoutException:
                # 若超時則跳過
                pass
        else:
            # 等待至少一筆資料
            WebDriverWait(self.driver, 20).until(
                EC.presence_of_all_elements_located((By.CSS_SELECTOR, self.EVENT_ROW_SELECTOR))
            )

    def _load_all_events(self) -> None:
        """
        不斷往下滾動並嘗試點擊 "Show More"，直到資料行數不再增加為止。
        """
        last_count = len(self.driver.find_elements(By.CSS_SELECTOR, self.EVENT_ROW_SELECTOR))
        while True:
            # 滾動到最後一行觸發自動載入
            self.driver.execute_script(
                f"const rows = document.querySelectorAll('{self.EVENT_ROW_SELECTOR}');" 
                "if(rows.length) rows[rows.length-1].scrollIntoView();"
            )
            time.sleep(1)
            new_count = len(self.driver.find_elements(By.CSS_SELECTOR, self.EVENT_ROW_SELECTOR))
            if new_count > last_count:
                last_count = new_count
                continue
            # 嘗試點擊 "Show More" 按鈕
            try:
                more = self.driver.find_element(By.CSS_SELECTOR, self.SHOW_MORE_SELECTOR)
                if more.is_displayed():
                    more.click()
                    time.sleep(1)
                    new_count = len(self.driver.find_elements(By.CSS_SELECTOR, self.EVENT_ROW_SELECTOR))
                    if new_count > last_count:
                        last_count = new_count
                        continue
                break
            except NoSuchElementException:
                break

    def _parse_events(self) -> pd.DataFrame:
        """
        將最終頁面 HTML 解析成 DataFrame，包含：
          - Time: 從 data-event-datetime 屬性取得
          - Country: 從對應 <span title> 取得
          - Impact: 從 sentiment td 的 title 取得
          - Event / Actual / Forecast / Previous: 文字內容
        """
        soup = BeautifulSoup(self.driver.page_source, "lxml")
        records = []
        for row in soup.select(self.EVENT_ROW_SELECTOR):
            cells = row.find_all("td")
            records.append({
                "Time": row.get("data-event-datetime", ""),
                "Country": (
                    row.select_one("td.flagCur.noWrap span").get("title", "")
                    if row.select_one("td.flagCur.noWrap span") else ""
                ),
                "Impact": (
                    row.select_one("td.sentiment").get("title", "")
                    if row.select_one("td.sentiment") else ""
                ),
                "Event": (
                    row.select_one("td.event a").get_text(strip=True)
                    if row.select_one("td.event a") else ""
                ),
                "Actual": cells[4].get_text(strip=True) if len(cells) > 4 else "",
                "Forecast": cells[5].get_text(strip=True) if len(cells) > 5 else "",
                "Previous": cells[6].get_text(strip=True) if len(cells) > 6 else "",
            })
        return pd.DataFrame(records)

    def scrape(self) -> pd.DataFrame:
        """
        主流程：
          1. 關閉 cookie 橫幅
          2. 設定時區 GMT+8
          3. 篩選國家
          4. 篩選日期
          5. 滾動並載入所有事件
          6. 解析成 DataFrame
        """
        self._close_cookie_banner()
        self._set_timezone_gmt8()
        self._apply_country_filter()
        self._apply_date_filter()
        self._load_all_events()
        df = self._parse_events()
        self.driver.quit()
        return df


if __name__ == "__main__":
    # 使用範例：抓取 2025-04-01 至 2025-04-10 的美國事件 (value="5")
    scraper = InvestingCalendarScraper(
        start_date="2025-04-01",
        end_date="2025-04-10",
        country_ids=["5"]
    )
    df = scraper.scrape()
    print(f"共抓到 {len(df)} 筆資料")
    df.to_csv('investing_calendar.csv')


共抓到 152 筆資料
